In [1]:
import pandas as pd
import numpy as np

from MATRIX.models import BaseSurvival, CoxRegression, DeepSurv, RandomSurvForest, DeepMultiTask, CoxRegressionWithTimeVarying, DeepTimeVarying
from MATRIX.utils import get_data, get_metrics, get_results, get_xai, save_results

# Experiments

## Survival analysis (standard)

### Get data

In [2]:
X_train_standard, y_train_standard, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv")


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [3]:
modelCoxRegression = CoxRegression()
modelRandomSurvForest = RandomSurvForest(0)
modelDeepSurv = DeepSurv(X_train_standard.shape[1])

### Fit and predict

In [4]:
modelCoxRegression = modelCoxRegression.fit(X_train_standard, y_train_standard)
modelRandomSurvForest = modelRandomSurvForest.fit(X_train_standard, y_train_standard)
modelDeepSurv = modelDeepSurv.fit(X_train_standard, y_train_standard)

survPredCoxRegression = modelCoxRegression.predict(X_test)
survPredRandomSurvForest = modelRandomSurvForest.predict(X_test)
survPredDeepSurv = modelDeepSurv.predict(X_test)

### Metrics

In [5]:
get_metrics([y_train_standard, y_test], [survPredCoxRegression])

{'C-Index Harrel': 0.6011437635665386,
 'C-Index IPCW': 0.6102418142329669,
 'Cumulative Dinamic AUC': [0.5832670650305211,
  0.6284814887997692,
  0.6584076144438055]}

In [6]:
get_metrics([y_train_standard, y_test], [survPredRandomSurvForest])

{'C-Index Harrel': 0.5958841208882952,
 'C-Index IPCW': 0.598702568660489,
 'Cumulative Dinamic AUC': [0.5788004074123053,
  0.6242527737906206,
  0.6542207854260769]}

In [7]:
get_metrics([y_train_standard, y_test], [survPredDeepSurv])

{'C-Index Harrel': 0.46873434630155286,
 'C-Index IPCW': 0.44796871536857497,
 'Cumulative Dinamic AUC': [0.474992488625633,
  0.43369477801024414,
  0.4333897780332707]}

### Survival / Cumulative Hazard Functions

In [ ]:
survival_function = modelCoxRegression.predict_survival_function(X=X_train_standard, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=False)

In [ ]:
cumulative_hazard_function = modelCoxRegression.predict_cumulative_hazard_function(X=X_train_standard, estimator_name="modelCoxRegression", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [ ]:
modelCoxRegression.calculate_xai(X=X_train_standard, estimator_name="modelCoxRegression", dataset="colon.csv", seed=None, feature_names=feature_names, background=50, plot=False)

## Survival analysis (multitask)

### Get data

In [11]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(dataset_name="colon.csv", to_multitask=True)


- - - - colon.csv (csv) - - - -

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event         911 non-null    int64  
 1   time          911 non-null    int64  
 2   num_age       911 non-null    int64  
 3   num_nodes     911 non-null    float64
 4   fac_rx        911 non-null    object 
 5   fac_sex       911 non-null    int64  
 6   fac_differ    911 non-null    object 
 7   fac_obstruct  911 non-null    int64  
 8   fac_perfor    911 non-null    int64  
 9   fac_adhere    911 non-null    int64  
 10  fac_node4     911 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 78.4+ KB

             event         time     num_age   num_nodes fac_rx     fac_sex  \
count   911.000000   911.000000  911.000000  911.000000    911  911.000000   
unique         NaN          NaN         NaN         NaN      3         NaN   
top 

### Instantiate models

In [12]:
modelDeepMultiTask = DeepMultiTask(X_train.shape[1])

### Fit and predict

In [13]:
modelDeepMultiTask = modelDeepMultiTask.fit(X_train, y_train)
survPredDeepMultiTask, binPredDeepMultiTask = modelDeepMultiTask.predict(X_test)

### Metrics

In [14]:
get_metrics([y_train, y_test], [survPredDeepMultiTask, binPredDeepMultiTask])

{'C-Index Harrel': 0.5150692937051261,
 'C-Index IPCW': 0.5077993684864526,
 'Cumulative Dinamic AUC': [0.5164806513956655,
  0.5102703562503049,
  0.5331083440800005],
 'MAE': 0.5191256830601093,
 'AMAE': 0.5191256830601093,
 'MS': 0.16939890710382513,
 'CCR': 0.4808743169398907,
 'RECALL0': 0.3114754098360656,
 'RECALL1': 0.16939890710382513}

### Survival / Cumulative Hazard Functions

In [15]:
survival_function = modelDeepMultiTask.predict_survival_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

In [16]:
cumulative_hazard_function = modelDeepMultiTask.predict_cumulative_hazard_function(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [17]:
modelDeepMultiTask.calculate_xai(X=X_train, estimator_name="modelDeepMultiTask", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

## Survival analysis (time varying)

In [2]:
df = pd.read_csv("./MATRIX/datasets/colon.csv")

### Find splits dynamically

In [4]:
splits = BaseSurvival.dinamic_discretise(y=df[["time", "event"]], dataset="colon", seed=0, plot=False)

In [5]:
df["identifier"] = df.index.values

### Transform to time varying format

In [6]:
df = BaseSurvival.to_time_dependent(dataframe=df, splits=splits, identifier="identifier", time="time", event="event")

In [9]:
df = BaseSurvival.to_time_varying(dataframe=df, identifier="identifier", time="time", event="event")

### Get data

In [23]:
X_train, y_train, X_validation, y_validation, X_test, y_test, feature_names, scaler = get_data(df=df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4049 entries, 0 to 4048
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   identifier    4049 non-null   int64  
 1   num_age       4049 non-null   int64  
 2   num_nodes     4049 non-null   float64
 3   fac_rx        4049 non-null   object 
 4   fac_sex       4049 non-null   int64  
 5   fac_differ    4049 non-null   object 
 6   fac_obstruct  4049 non-null   int64  
 7   fac_perfor    4049 non-null   int64  
 8   fac_adhere    4049 non-null   int64  
 9   fac_node4     4049 non-null   int64  
 10  event         4049 non-null   int64  
 11  time_start    4049 non-null   float64
 12  time_stop     4049 non-null   float64
dtypes: float64(3), int64(8), object(2)
memory usage: 411.4+ KB

         identifier      num_age    num_nodes   fac_rx      fac_sex  \
count   4049.000000  4049.000000  4049.000000     4049  4049.000000   
unique          NaN          NaN         

### Instantiate models

In [24]:
modelCoxRegressionWithTimeVarying = CoxRegressionWithTimeVarying()
modelDeepTimeVarying = DeepTimeVarying(X_train.shape[1])

### Fit and predict

In [25]:
modelCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.fit(X_train, y_train)
modelDeepTimeVarying = modelDeepTimeVarying.fit(X_train, y_train)

survPredCoxRegressionWithTimeVarying = modelCoxRegressionWithTimeVarying.predict(X_test)
survPredDeepTimeVarying = modelDeepTimeVarying.predict(X_test)

### Metrics

In [26]:
get_metrics([y_train, y_test], [survPredCoxRegressionWithTimeVarying])

{'C-Index Harrel': 0.6706086623646506,
 'C-Index IPCW': 0.6585322447399417,
 'Cumulative Dinamic AUC': [0.6770340633108334,
  0.6695773476164759,
  0.6192424454026013]}

In [27]:
get_metrics([y_train, y_test], [survPredDeepTimeVarying])

{'C-Index Harrel': 0.5069452039811878,
 'C-Index IPCW': 0.5101314470110674,
 'Cumulative Dinamic AUC': [0.46800686588479573,
  0.5522360203619334,
  0.5214476025548159]}

### Survival / Cumulative Hazard Functions

In [28]:
survival_function = modelCoxRegressionWithTimeVarying.predict_survival_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

In [29]:
cumulative_hazard_function = modelCoxRegressionWithTimeVarying.predict_cumulative_hazard_function(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, plot=False)

### eXplainable Artificial Intelligence

In [30]:
modelCoxRegressionWithTimeVarying.calculate_xai(X=X_train, estimator_name="modelCoxRegressionWithTimeVarying", dataset="colon.csv", seed=0, feature_names=feature_names, background=50, plot=False)

## Tools

### Feature selector

In [23]:
features_selected = BaseSurvival.feature_selection(X_train_standard, y_train_standard)

In [34]:
print(f"ALL FEATURES: {X_train_standard.shape}, SELECTED FEATURES: {X_train_standard[:, features_selected].shape}")

ALL FEATURES: (582, 6), SELECTED FEATURES: (582, 4)


### Simulated survival data

In [39]:
BaseSurvival.generate_simulated_survival_data(number_rows=1000, number_columns=10, censored=0.75, relation="sin", seed=0)

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,event,time
0,1.764052,0.400157,0.978738,2.240893,1.867558,-0.977278,0.950088,-0.151357,-0.103219,0.410599,0,7.39
1,0.144044,1.454274,0.761038,0.121675,0.443863,0.333674,1.494079,-0.205158,0.313068,-0.854096,1,82.31
2,-2.552990,0.653619,0.864436,-0.742165,2.269755,-1.454366,0.045759,-0.187184,1.532779,1.469359,1,5.16
3,0.154947,0.378163,-0.887786,-1.980796,-0.347912,0.156349,1.230291,1.202380,-0.387327,-0.302303,1,3.00
4,-1.048553,-1.420018,-1.706270,1.950775,-0.509652,-0.438074,-1.252795,0.777490,-1.613898,-0.212740,0,2.93
...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.622801,1.240477,0.170446,1.986898,0.111301,1.209655,-0.457330,0.923183,1.325519,-0.953166,0,0.53
996,-0.678130,0.334100,-0.754928,-0.388383,-0.626576,-1.306421,-0.038496,0.143567,-0.789074,-0.098504,0,0.09
997,-1.452866,-0.947866,-0.869579,-0.305517,-0.605147,0.224821,-1.038472,0.771819,-0.480009,1.834745,0,0.06
998,-0.108765,0.454475,0.098709,-0.458305,-0.857044,-0.774403,0.708787,0.018474,-0.104081,-0.234858,0,0.33


### Log-rank test

In [11]:
identifier = np.arange(len(X_train_standard))

In [12]:
# H0 = "There is no difference in the survival curves between the groups analysed" (p-value < threshold ==> reject H0)
BaseSurvival.logrank_test(y_train_standard, identifier)

<lifelines.StatisticalResult: multivariate_logrank_test>
               t_0 = -1
 null_distribution = chi squared
degrees_of_freedom = 581
         test_name = multivariate_logrank_test

---
 test_statistic      p  -log2(p)
        3459.80 <0.005       inf

# Results

## Show stored results

In [5]:
results = get_results(estimator_name="CoxRegression", dataset="colon.csv")
#results = get_results(estimator_name="CoxRegressionWithTimeVarying", dataset="cgd.csv")
#results = get_results(estimator_name="DeepMultiTaskFFNN", dataset="colon.csv")

In [4]:
for i, result in enumerate(results):
    print(f"Result {i}:\n")
    print(f"    * Estimator: {result.config['estimator_name']} - Dataset: {result.config['dataset']} - Seed: {result.config['random_state']}")
    print(f"    * Best Params: {result.data_.best_params}")
    print()
    print(f"    * Metrics: {get_metrics(result.data_.targets, result.data_.predictions)}")
    print("\n\n")

Result 0:

    * Estimator: CoxRegression - Dataset: colon.csv - Seed: 0
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 10.0}

    * Metrics: {'C-Index Harrel': 0.6005593588245116, 'C-Index IPCW': 0.6171115091088967, 'Cumulative Dinamic AUC': [0.5790226380142177, 0.6288561402862889, 0.6565738162897639]}



Result 1:

    * Estimator: CoxRegression - Dataset: colon.csv - Seed: 1
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 10.0}

    * Metrics: {'C-Index Harrel': 0.6278135048231511, 'C-Index IPCW': 0.6344141780648662, 'Cumulative Dinamic AUC': [0.6651978973407545, 0.6484380362536082, 0.7288120768413935]}



Result 2:

    * Estimator: CoxRegression - Dataset: colon.csv - Seed: 2
    * Best Params: {'ties': 'efron', 'n_iter': 100, 'alpha': 0.1}

    * Metrics: {'C-Index Harrel': 0.6677913854869616, 'C-Index IPCW': 0.6438962832758339, 'Cumulative Dinamic AUC': [0.6921737644102026, 0.7141277809777058, 0.7134722573314778]}



Result 3:

    * Estimator: CoxRegr

## Plot XAI results 

In [7]:
get_xai(estimator_name="CoxRegression", dataset="colon.csv", seed=None, individual=0)
#get_xai(estimator_name="CoxRegression", dataset="colon.csv")
#get_xai(dataset="colon.csv")

## Save stored results 

In [2]:
save_results(estimator_name="CoxRegression", dataset="aids2.csv")